In [1]:
# Step 1: Set Up Your Environment
from pyspark.sql import SparkSession

In [2]:
# Initialize Spark session
spark=SparkSession.builder.appName("dogfood").getOrCreate()

In [18]:
# Step 2: Load the Data
data=spark.read.csv("dog_food.csv", header=True, inferSchema=True)
data.show(3)

+---+---+----+---+-------+
|  A|  B|   C|  D|Spoiled|
+---+---+----+---+-------+
|  4|  2|12.0|  3|    1.0|
|  5|  6|12.0|  7|    1.0|
|  6|  2|13.0|  6|    1.0|
+---+---+----+---+-------+
only showing top 3 rows



In [4]:
# Step 3: Data Preprocessing
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

In [5]:
# Example: Encoding a categorical column
indexer = StringIndexer(inputCol="Spoiled", outputCol="label")

In [6]:
# Example: Assembling feature columns into a feature vector
assembler = VectorAssembler(inputCols=["A","B","C","D"],
outputCol="features")

In [7]:
# Create a pipeline for preprocessing
pipeline = Pipeline(stages=[indexer, assembler])
data_preprocessed = pipeline.fit(data).transform(data)

In [8]:
# Split the data into training and testing sets
train_data, test_data = data_preprocessed.randomSplit([0.8, 0.2], seed=1)

In [9]:
# Step 4: Build the Classification Model
from pyspark.ml.classification import LogisticRegression
# Initialize the classifier
lr = LogisticRegression(featuresCol="features", labelCol="label")
# Train the model
lr_model = lr.fit(train_data)

In [10]:
# Step 5: Evaluate the Model
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# Make predictions
predictions = lr_model.transform(test_data)
# Evaluate the model
evaluator = MulticlassClassificationEvaluator(labelCol="label",
predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Test Accuracy = {accuracy}")

Test Accuracy = 0.9891304347826086


In [17]:
from pyspark.sql.functions import col

# Compute Confusion Matrix
confusion_matrix = predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
confusion_matrix.show()

# Extract TP, TN, FP, FN manually
tp = predictions.filter((col("label") == 1.0) & (col("prediction") == 1.0)).count()
tn = predictions.filter((col("label") == 0.0) & (col("prediction") == 0.0)).count()
fp = predictions.filter((col("label") == 0.0) & (col("prediction") == 1.0)).count()
fn = predictions.filter((col("label") == 1.0) & (col("prediction") == 0.0)).count()

# Print Confusion Matrix
print(f"Confusion Matrix:\n TP: {tp}, FP: {fp}\n FN: {fn}, TN: {tn}")


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|   69|
|  1.0|       0.0|    1|
|  1.0|       1.0|   22|
+-----+----------+-----+

Confusion Matrix:
 TP: 22, FP: 0
 FN: 1, TN: 69


In [ ]:
# Step 6: Save the Model 
lr_model.save(r"/work/kripa/")